# Carga de imagenes al datastore y mosaic dataset

Flujo operativo posterior a la preparacion del notebook `008`. Usa `04_ready_for_datastore.csv` para copiar archivos al datastore y `06_attribute_updates.csv` para actualizar atributos del mosaic dataset.

In [1]:
from datetime import datetime
from pathlib import Path
import importlib

import pandas as pd

import core.mosaic_loader as mosaic_loader
mosaic_loader = importlib.reload(mosaic_loader)
from core.mosaic_loader import *

# PARAMETROS
RUN_PREPARACION = "20260615_155625"
OUTPUT_PREPARACION_DIR = Path.cwd() / "outputs" / "preparacion_carga_mosaico" / RUN_PREPARACION

READY_FOR_DATASTORE_CSV = OUTPUT_PREPARACION_DIR / "04_ready_for_datastore.csv"
ATTRIBUTE_UPDATES_CSV = OUTPUT_PREPARACION_DIR / "06_attribute_updates.csv"

PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"

# Seguridad operacional: validar primero con DRY_RUN=True. Cambiar a False para ejecutar copia/carga/update.
DRY_RUN = True
OVERWRITE_COPY = False
SKIP_EXISTING_MOSAIC_NAME = True

# Valores fijos definidos para esta carga.
MAXPS_VALUE = 10000
LOWPS_VALUE = 0.15

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path.cwd() / "outputs" / "carga_mosaico" / run_timestamp
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Preparacion:", OUTPUT_PREPARACION_DIR)
print("CSV carga:", READY_FOR_DATASTORE_CSV)
print("CSV atributos:", ATTRIBUTE_UPDATES_CSV)
print("Mosaic dataset:", PATH_MOSAIC_DATASET)
print("Salida:", OUTPUT_DIR)
print("DRY_RUN:", DRY_RUN)

Preparacion: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\preparacion_carga_mosaico\20260615_155625
CSV carga: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\preparacion_carga_mosaico\20260615_155625\04_ready_for_datastore.csv
CSV atributos: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\preparacion_carga_mosaico\20260615_155625\06_attribute_updates.csv
Mosaic dataset: \\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport
Salida: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\carga_mosaico\20260615_162243
DRY_RUN: True


## 1. Cargar manifiestos

Se valida que cada imagen lista tenga atributos asociados antes de ejecutar cualquier operacion.

In [2]:
load_df = load_ready_and_attributes(READY_FOR_DATASTORE_CSV, ATTRIBUTE_UPDATES_CSV)

required_columns = ["path", "destination_path", "Name", "Sector", "Fecha_Adqui", "URL", "Proyecto", "Sensor", "Fecha_Publ"]
missing_columns = [column for column in required_columns if column not in load_df.columns]
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {missing_columns}")

validation_summary = pd.DataFrame(
    [
        {"metric": "records_to_process", "value": len(load_df)},
        {"metric": "missing_source_path", "value": int((~load_df["path"].map(lambda value: Path(value).exists())).sum())},
        {"metric": "missing_destination_path", "value": int(load_df["destination_path"].isna().sum())},
        {"metric": "unique_destination_paths", "value": int(load_df["destination_path"].nunique())},
        {"metric": "unique_names", "value": int(load_df["Name"].nunique())},
    ]
)

display(validation_summary)
display(load_df[["file_name", "Name", "destination_path", "Sector", "Fecha_Adqui", "Proyecto", "Sensor", "Fecha_Publ"]].head(20))

,metric,value
0,records_to_process,32
1,missing_source_path,0
2,missing_destination_path,0
3,unique_destination_paths,32
4,unique_names,32


,file_name,Name,destination_path,Sector,Fecha_Adqui,Proyecto,Sensor,Fecha_Publ
0,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...,CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_D...,Estacion_Cabecera,2026-05-01,PAO,DJI Mavic Enterprise,2026-06-15
1,GEOSP-TRN-002545_GS_ORTOFOTO_EB3_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,Subestacion-El-Mauro,2026-05-06,PAO,DJI Mavic Enterprise,2026-06-15
2,GEOSP-TRN-002546_GS_ORTOFOTO_SSEE_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,Subestacion-El-Mauro,2026-05-06,PAO,DJI Mavic Enterprise,2026-06-15
3,GEOSP-TRN-002555_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,TORRE_E85_A_E_125,2026-05-06,PAO,DJI Mavic Enterprise,2026-06-15
4,GEOSP-TRN-002591_GS_Ortofoto ED1_10-05-2026.tif,CL_MLP_PAO_IF_Ortho_26_05_10_ED1,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ED1,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15
5,GEOSP-TRN-002592_GS_Ortofoto_Helipuerto Mauro ...,CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,Helipuerto,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15
6,GEOSP-TRN-002593_GS_ORTOFOTO_PATIO 19B_10-05-2...,CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,Patio-19B-y-Armado,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15
7,GEOSP-TRN-002603_ORTOFOTO_CORTADA_EM2_100526.tif,CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,DME9-PA12-IIFF8,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15
8,GEOSP-TRN-002604_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,TORRES_E48_A_E84_PV4,2026-05-07,PAO,DJI Mavic Enterprise,2026-06-15
9,GEOSP-TRN-002606_GS_Ortofoto_Tramo 1 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15


## 2. Ejecutar copia, carga al mosaico, footprints y atributos

Con `DRY_RUN=True` no escribe archivos ni modifica el mosaic dataset. Con `DRY_RUN=False` ejecuta el flujo completo por imagen.

In [3]:
results = []

for index, row in load_df.iterrows():
    print(f"[{index + 1}/{len(load_df)}] {row['Name']}")
    result = process_mosaic_load_row(
        row,
        PATH_MOSAIC_DATASET,
        overwrite_copy=OVERWRITE_COPY,
        skip_existing_mosaic_name=SKIP_EXISTING_MOSAIC_NAME,
        maxps_value=MAXPS_VALUE,
        lowps_value=LOWPS_VALUE,
        dry_run=DRY_RUN,
    )
    results.append(result)

results_df = pd.DataFrame(results)
display(results_df)
display(results_df["overall_status"].value_counts(dropna=False).reset_index(name="count").rename(columns={"index": "overall_status"}))

[1/32] CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera
[2/32] CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-1
[3/32] CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-2
[4/32] CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125
[5/32] CL_MLP_PAO_IF_Ortho_26_05_10_ED1
[6/32] CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto
[7/32] CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado
[8/32] CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8
[9/32] CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4
[10/32] CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-Km-84p2-a-82p3
[11/32] CL_MLP_PAO_IF_Ortho_26_05_13_DME9-PA12-IIFF8
[12/32] CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro_A_E35
[13/32] CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-1
[14/32] CL_MLP_PAO_IF_Ortho_26_05_13_EM2_S2
[15/32] CL_MLP_PAO_IF_Ortho_26_05_13_EBD-1
[16/32] CL_MLP_PAO_IF_Ortho_26_05_13_ED2
[17/32] CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-2
[18/32] CL_MLP_PAO_IF_Ortho_26_05_13_EBD-2
[19/32] CL_MLP_PAO_IF_Ortho_26_05_13_Estacion_

,file_name,Name,destination_path,overall_status,source_path,copy_status,copy_error,mosaic_add_status,mosaic_add_error,footprint_status,footprint_error,attribute_status,attribute_rows_updated,attribute_missing_fields,attribute_error
0,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...,CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_D...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,dry_run,None,dry_run,None,dry_run,None,dry_run,0,None,None
1,GEOSP-TRN-002545_GS_ORTOFOTO_EB3_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,dry_run,None,dry_run,None,dry_run,None,dry_run,0,None,None
2,GEOSP-TRN-002546_GS_ORTOFOTO_SSEE_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,dry_run,None,dry_run,None,dry_run,None,dry_run,0,None,None
3,GEOSP-TRN-002555_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,dry_run,None,dry_run,None,dry_run,None,dry_run,0,None,None
4,GEOSP-TRN-002591_GS_Ortofoto ED1_10-05-2026.tif,CL_MLP_PAO_IF_Ortho_26_05_10_ED1,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,dry_run,None,dry_run,None,dry_run,None,dry_run,0,None,None
5,GEOSP-TRN-002592_GS_Ortofoto_Helipuerto Mauro ...,CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,dry_run,None,dry_run,None,dry_run,None,dry_run,0,None,None
6,GEOSP-TRN-002593_GS_ORTOFOTO_PATIO 19B_10-05-2...,CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,dry_run,None,dry_run,None,dry_run,None,dry_run,0,None,None
7,GEOSP-TRN-002603_ORTOFOTO_CORTADA_EM2_100526.tif,CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,dry_run,None,dry_run,None,dry_run,None,dry_run,0,None,None
8,GEOSP-TRN-002604_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,dry_run,None,dry_run,None,dry_run,None,dry_run,0,None,None
9,GEOSP-TRN-002606_GS_Ortofoto_Tramo 1 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,dry_run,None,dry_run,None,dry_run,None,dry_run,0,None,None


,overall_status,count
0,dry_run,32


## 3. Exportar resultados de ejecucion

In [4]:
summary_rows = [
    {"metric": "run_timestamp", "value": run_timestamp},
    {"metric": "preparation_run", "value": RUN_PREPARACION},
    {"metric": "dry_run", "value": DRY_RUN},
    {"metric": "mosaic_dataset", "value": PATH_MOSAIC_DATASET},
    {"metric": "records_to_process", "value": len(load_df)},
    {"metric": "maxps_value", "value": MAXPS_VALUE},
    {"metric": "lowps_value", "value": LOWPS_VALUE},
]

for column in ["copy_status", "mosaic_add_status", "footprint_status", "attribute_status", "overall_status"]:
    if column in results_df.columns:
        for status, count in results_df[column].value_counts(dropna=False).items():
            summary_rows.append({"metric": f"{column}_{status}", "value": int(count)})

summary_df = pd.DataFrame(summary_rows)

summary_csv = OUTPUT_DIR / "00_summary.csv"
results_csv = OUTPUT_DIR / "01_load_results.csv"
errors_csv = OUTPUT_DIR / "02_errors.csv"

summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
results_df.to_csv(results_csv, index=False, encoding="utf-8-sig")
error_columns = [column for column in results_df.columns if column.endswith("_error")]
error_filter = results_df[error_columns].notna().any(axis=1) if error_columns else pd.Series(False, index=results_df.index)
results_df[error_filter].to_csv(errors_csv, index=False, encoding="utf-8-sig")

display(summary_df)
print("Resultados exportados en:", OUTPUT_DIR)

,metric,value
0,run_timestamp,20260615_162243
1,preparation_run,20260615_155625
2,dry_run,True
3,mosaic_dataset,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
4,records_to_process,32
5,maxps_value,10000
6,lowps_value,0.15
7,copy_status_dry_run,32
8,mosaic_add_status_dry_run,32
9,footprint_status_dry_run,32


Resultados exportados en: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\carga_mosaico\20260615_162243
